# 🧠 NeuroSleepNet — Interactive Demo

> **Give local LLMs infinite memory.**

This notebook demonstrates NeuroSleepNet — a lightweight, zero-ops memory layer for small, open-source LLMs. It runs entirely locally, requires **zero external databases**, and augments any model with persistent memory in just **2 lines of code**.

### What You'll See
1. **Round 1** — A bare Qwen-0.5B model fails to recall personal facts (it's stateless)
2. **Round 2** — The same model wrapped with NeuroSleepNet correctly recalls everything
3. **Round 3** — Long multi-turn conversation memory retrieval
4. **Scorecard** — Side-by-side accuracy comparison

### Architecture Highlights
- **Hybrid BM25 + Semantic Search** — Fuses keyword matching with dense vector similarity
- **Time-Decay Recency Weighting** — Newest facts automatically outrank stale ones
- **LRU Edge Cache** — Sub-5ms retrieval, no network hops
- **Knowledge Graph Memory** — Entity-relationship extraction for structured recall
- **Sleep Consolidator** — Periodic memory deduplication and noise pruning

---

⚡ **Runtime:** Free-tier Colab (CPU or T4 GPU). Total run time: ~3–5 minutes.

📦 **GitHub:** [github.com/avirooppal/NeuroSleepNet](https://github.com/avirooppal/NeuroSleepNet)

---
## 📦 Step 1 — Install Dependencies

Install the HuggingFace stack and NeuroSleepNet directly from GitHub.

In [ ]:
!pip install -q transformers torch accelerate
!pip install -q git+https://github.com/avirooppal/NeuroSleepNet.git
print("\n✅ All dependencies installed.")

---
## 🤖 Step 2 — Load a Local LLM

We load **Qwen2.5-0.5B-Instruct** — a ~1 GB instruction-tuned model that runs comfortably on free-tier Colab. This model is completely **stateless**: each `.predict()` call only sees the current prompt.

In [ ]:
import time
import textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

class LocalLLMAgent:
    """
    A real (small) LLM agent running locally.
    Each .predict() call is STATELESS — the model receives only
    what's in the current prompt. This is the problem NeuroSleepNet solves.
    """
    def __init__(self, model_id: str = MODEL_ID):
        print(f"⏳ Loading model: {model_id} ...")
        t0 = time.time()
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            trust_remote_code=True,
            device_map="auto",
        )
        self.pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
        )
        print(f"✅ Model loaded in {time.time() - t0:.1f}s")

    def predict(self, prompt: str, **kwargs) -> str:
        messages = [
            {"role": "system", "content": "You are a helpful personal assistant. Answer concisely in 1-2 sentences based ONLY on what is provided in the prompt. If you don't have the information, say 'I don't have that information.'"},
            {"role": "user", "content": prompt},
        ]
        output = self.pipe(
            messages,
            max_new_tokens=100,
            do_sample=False,
            temperature=None,
            top_p=None,
        )
        return output[0]["generated_text"][-1]["content"].strip()


base_agent = LocalLLMAgent()

---
## 🛑 Round 1 — Bare LLM (No Memory)

We ask the model 5 personal questions it has **never** seen the answers to. Since it's stateless, it can't recall any user-specific facts.

In [ ]:
# ── Helper ──────────────────────────────────────────────────────────────────
def ask(agent, question, label, task_id=None):
    print(f"  ❓ {question}")
    t0 = time.perf_counter()
    if task_id is not None:
        answer = agent.predict(question, task_id=task_id)
    else:
        answer = agent.predict(question)
    ms = (time.perf_counter() - t0) * 1000
    wrapped = textwrap.fill(answer, width=72, initial_indent="     ", subsequent_indent="     ")
    print(f"  💬 [{label}] ({ms:.0f} ms):\n{wrapped}\n")
    return answer


# ── Personal facts (the model has NEVER seen these) ────────────────────────
facts = [
    "My dog's name is Biscuit and he is a golden retriever.",
    "I am severely allergic to shellfish — no shrimp, crab, or lobster.",
    "My wedding anniversary is on March 14th.",
    "I drive a midnight-blue 2022 Tesla Model 3.",
    "My favourite programming language is Rust.",
]

questions = [
    "What is my dog's name and breed?",
    "I'm thinking of ordering shrimp for dinner. Is that okay for me?",
    "When is my wedding anniversary?",
    "What car do I drive?",
    "Which programming language do I like the most?",
]


# ── Round 1 ─────────────────────────────────────────────────────────────────
print("═" * 64)
print("  🛑 ROUND 1: Bare LLM — No Memory")
print("═" * 64)
print("  The model receives ONLY the question. It has never seen your facts.\n")

bare_answers = []
for q in questions:
    ans = ask(base_agent, q, "Bare")
    bare_answers.append(ans)

---
## ✅ Round 2 — LLM + NeuroSleepNet Sidecar

Now we wrap the **exact same model** with NeuroSleepNet. Two lines of code:

```python
from neurosleepnet import wrap
memory_agent = wrap(base_agent, mode="sidecar")
```

We teach it 5 personal facts, then ask the same questions. NeuroSleepNet silently injects relevant memories into each prompt using **hybrid search + knowledge graph + recency weighting**.

In [ ]:
from neurosleepnet import wrap

# ── Wrap with NeuroSleepNet (THIS IS THE MAGIC — 2 lines!) ─────────────────
memory_agent = wrap(base_agent, mode="sidecar")

print("═" * 64)
print("  ✅ ROUND 2: LLM + NeuroSleepNet Sidecar")
print("═" * 64)

# ── Teach the agent ────────────────────────────────────────────────────────
print("\n  📚 Teaching the agent 5 personal facts...\n")
for i, fact in enumerate(facts, 1):
    memory_agent.learn(
        task_id="demo_session",
        input_data=fact,
        label="PersonalFact",
    )
    print(f"    [{i}] ✓ {fact}")

# ── Ask the SAME questions ─────────────────────────────────────────────────
print("\n  Now asking the SAME questions.")
print("  NeuroSleepNet silently injects relevant memories into each prompt.\n")

nsn_answers = []
for q in questions:
    ans = ask(memory_agent, q, "NSN+LLM", task_id="demo_session")
    nsn_answers.append(ans)

---
## 🌐 Round 3 — Multi-Turn Long Context Memory

Simulating a 14-turn conversation. A normal LLM would need **all 14 turns re-sent** in every prompt (blowing up tokens and latency). NeuroSleepNet retrieves only what's relevant.

In [ ]:
print("═" * 64)
print("  🌐 ROUND 3: Multi-Round Long Context Memory")
print("═" * 64)

conversation_history = [
    "User: Hi, my name is Alex and I'm a software engineer.",
    "Agent: Nice to meet you Alex! What kind of software do you write?",
    "User: I do a lot of Python backend work, usually with FastAPI.",
    "Agent: That's great, FastAPI is very fast and modern.",
    "User: Yeah. By the way, I have a cat named Whiskers.",
    "Agent: Whiskers is a cute name! How old is he?",
    "User: He's 4 years old and loves salmon.",
    "Agent: Good to know. Do you have any other pets?",
    "User: No, just him. I used to have a parrot but he flew away.",
    "Agent: Oh no, I'm sorry to hear that.",
    "User: It's okay. Anyway, I'm planning a trip to Japan next month.",
    "Agent: Japan is beautiful! Which cities are you visiting?",
    "User: Kyoto and Tokyo. I really want to see the bamboo forests.",
    "Agent: That sounds like an amazing itinerary. Enjoy!",
]

print("\n  📚 Teaching the agent 14 conversation turns...\n")
for i, turn in enumerate(conversation_history, 1):
    memory_agent.learn(
        task_id="multi_turn_session",
        input_data=turn,
        label="ChatTurn"
    )
    if i <= 3 or i >= 13:
        print(f"    [{i}] ✓ {turn}")
    elif i == 4:
        print("    ... (more turns stored) ...")

print("\n  Asking questions about information scattered across long history.\n")

long_qs = [
    "What is my name and my profession?",
    "What pets do I have, and what does my pet like to eat?",
    "Where am I traveling to and what specific things do I want to see?",
]

for q in long_qs:
    ask(memory_agent, q, "NSN+LLM (Multi-Turn)", task_id="multi_turn_session")

---
## 📊 Scorecard — Bare vs. NeuroSleepNet

Automated accuracy check across all 5 personal fact questions.

In [ ]:
print("═" * 64)
print("  📊 SCORECARD")
print("═" * 64)

checks = [
    ("Dog = Biscuit / golden retriever", lambda a: "biscuit" in a.lower()),
    ("Shellfish allergy warning",        lambda a: "allerg" in a.lower() or "no" in a.lower() or "avoid" in a.lower() or "not" in a.lower()),
    ("Anniversary = March 14",           lambda a: "march" in a.lower() and "14" in a.lower()),
    ("Car = Tesla Model 3",              lambda a: "tesla" in a.lower() or "model 3" in a.lower()),
    ("Language = Rust",                   lambda a: "rust" in a.lower()),
]

bare_score = 0
nsn_score = 0

print(f"\n  {'Check':<38} {'Bare':>6} {'NSN':>6}")
print(f"  {'─' * 38} {'─' * 6} {'─' * 6}")

for (label, fn), ba, na in zip(checks, bare_answers, nsn_answers):
    b = fn(ba)
    n = fn(na)
    bare_score += int(b)
    nsn_score += int(n)
    print(f"  {label:<38} {'✅' if b else '❌':>6} {'✅' if n else '❌':>6}")

print(f"\n  {'TOTAL':<38} {bare_score}/5{'':>2} {nsn_score}/5")

if nsn_score > bare_score:
    improvement = ((nsn_score - bare_score) / max(bare_score, 1)) * 100
    print(f"\n  🏆 NeuroSleepNet improved recall: {bare_score}/5 → {nsn_score}/5 (+{improvement:.0f}%)")
    print("  ✅ Real LLM forgetting prevented — proof complete!")
elif nsn_score == 5 and bare_score == 5:
    print("\n  ⚠️  Both scored perfectly — try more obscure facts.")
else:
    print("\n  ⚠️  Unexpected — inspect the raw answers above.")

---
## 🔬 Bonus: Peek Under the Hood

Let's inspect what NeuroSleepNet actually does when you call `.predict()`. It builds an augmented prompt with memory context from the hybrid search engine and knowledge graph.

In [ ]:
print("═" * 64)
print("  🔬 UNDER THE HOOD: What NeuroSleepNet Injects")
print("═" * 64)

# Manually call the internal injection to see the augmented prompt
test_query = "What car do I drive?"
augmented = memory_agent._inject_sidecar_prompt(test_query, task_id="demo_session")

print(f"\n  Original query: '{test_query}'")
print(f"\n  Augmented prompt sent to the LLM:\n")
print("  ┌" + "─" * 62 + "┐")
for line in augmented.split("\n"):
    print(f"  │ {line:<60} │")
print("  └" + "─" * 62 + "┘")

print("\n  📌 The LLM sees relevant memories pre-injected into the prompt.")
print("  📌 Retrieval overhead: typically < 5ms via the LRU edge cache.")
print("  📌 No retraining, no fine-tuning — just intelligent prompt augmentation.")

---
## 🧪 Try It Yourself!

Teach the agent your own facts and ask questions. Edit the cells below and run them.

In [ ]:
# ── Teach a custom fact ────────────────────────────────────────────────────
memory_agent.learn(
    task_id="playground",
    input_data="My favourite movie is Interstellar directed by Christopher Nolan.",
    label="PersonalFact",
)
print("✓ Fact stored!")

# ── Ask about it ───────────────────────────────────────────────────────────
answer = memory_agent.predict(
    "What's my favourite movie and who directed it?",
    task_id="playground"
)
print(f"\n💬 {answer}")

---

### 🔗 Links

- **GitHub**: [github.com/avirooppal/NeuroSleepNet](https://github.com/avirooppal/NeuroSleepNet)
- **Documentation**: See `README.md` and `quickstart.py` in the repo

---

*Built with ❤️ by the NeuroSleepNet team. © 2026*